# C11-neural-training — Practice p21 — Solution


**Type:** challenge · **Difficulty:** advanced · **Concepts:** softmax, cross-entropy-loss


One stable helper evaluates both the base and perturbed losses. The returned
gradient remains analytic; finite differences are used only as an external
directional audit.


In [ ]:
import numpy as np

def _stable_loss_p21(z,y):
    shifted=z-z.max(axis=1,keepdims=True)
    return float(np.mean(np.log(np.exp(shifted).sum(axis=1))-shifted[np.arange(z.shape[0]),y]))

def fused_ce_audit(logits,labels,directions,epsilon=1e-6):
    z=np.asarray(logits,dtype=np.float64); y=np.asarray(labels); V=np.asarray(directions,dtype=np.float64)
    if z.ndim!=2 or y.shape!=(z.shape[0],) or V.ndim!=3 or V.shape[1:]!=z.shape:
        raise ValueError("invalid shapes")
    shifted=z-z.max(axis=1,keepdims=True); e=np.exp(shifted); p=e/e.sum(axis=1,keepdims=True)
    loss=_stable_loss_p21(z,y); grad=p.copy(); grad[np.arange(len(y)),y]-=1; grad/=len(y)
    errors=np.empty(V.shape[0],dtype=np.float64)
    for k in range(V.shape[0]):
        analytic=float(np.sum(grad*V[k])); numeric=(_stable_loss_p21(z+epsilon*V[k],y)-_stable_loss_p21(z-epsilon*V[k],y))/(2*epsilon)
        errors[k]=abs(analytic-numeric)
    return {"loss":loss,"probabilities":p,"gradient":grad,"directional_errors":errors}

rng_p21=np.random.default_rng(20260804); Z_p21=rng_p21.normal(size=(5,4))*3; y_p21=rng_p21.integers(0,4,size=5); V_p21=rng_p21.normal(size=(6,5,4))
Z_p21+=np.array([10000.,-10000.,5000.,-5000.]); result_p21=fused_ce_audit(Z_p21,y_p21,V_p21)


### Answer check


In [ ]:
assert set(result_p21)=={"loss","probabilities","gradient","directional_errors"}
assert result_p21["probabilities"].shape==(5,4) and result_p21["gradient"].shape==(5,4) and result_p21["directional_errors"].shape==(6,)
assert np.isfinite(result_p21["loss"]) and np.all(np.isfinite(result_p21["probabilities"])) and np.all(np.isfinite(result_p21["gradient"]))
assert np.allclose(result_p21["probabilities"].sum(1),1,atol=1e-11,rtol=1e-9)
assert np.allclose(result_p21["gradient"].sum(1),0,atol=1e-11,rtol=1e-9)
assert np.all(result_p21["directional_errors"]<2e-6)
shifted_result_p21=fused_ce_audit(Z_p21+np.arange(5)[:,None]*37,y_p21,V_p21)
assert np.allclose(result_p21["probabilities"],shifted_result_p21["probabilities"],atol=1e-11,rtol=1e-9)
